<div style="position: relative;" class="flex flex-col"><div class="HTMLContent_html__0OZLp" data-track-load="description_content"><p>You are given an <code>m x n</code> <code>grid</code>. Each cell of <code>grid</code> represents a street. The street of <code>grid[i][j]</code> can be:</p>

<ul>
	<li><code>1</code> which means a street connecting the left cell and the right cell.</li>
	<li><code>2</code> which means a street connecting the upper cell and the lower cell.</li>
	<li><code>3</code> which means a street connecting the left cell and the lower cell.</li>
	<li><code>4</code> which means a street connecting the right cell and the lower cell.</li>
	<li><code>5</code> which means a street connecting the left cell and the upper cell.</li>
	<li><code>6</code> which means a street connecting the right cell and the upper cell.</li>
</ul>
<img alt="" style="width: 450px; height: 708px;" src="https://assets.leetcode.com/uploads/2020/03/05/main.png">
<p>You will initially start at the street of the upper-left cell <code>(0, 0)</code>. A valid path in the grid is a path that starts from the upper left cell <code>(0, 0)</code> and ends at the bottom-right cell <code>(m - 1, n - 1)</code>. <strong>The path should only follow the streets</strong>.</p>

<p><strong>Notice</strong> that you are <strong>not allowed</strong> to change any street.</p>

<p>Return <code>true</code><em> if there is a valid path in the grid or </em><code>false</code><em> otherwise</em>.</p>

<p>&nbsp;</p>
<p><strong class="example">Example 1:</strong></p>
<img alt="" style="width: 455px; height: 311px;" src="https://assets.leetcode.com/uploads/2020/03/05/e1.png">
<pre><strong>Input:</strong> grid = [[2,4,3],[6,5,2]]
<strong>Output:</strong> true
<strong>Explanation:</strong> As shown you can start at cell (0, 0) and visit all the cells of the grid to reach (m - 1, n - 1).
</pre>

<p><strong class="example">Example 2:</strong></p>
<img alt="" style="width: 455px; height: 293px;" src="https://assets.leetcode.com/uploads/2020/03/05/e2.png">
<pre><strong>Input:</strong> grid = [[1,2,1],[1,2,1]]
<strong>Output:</strong> false
<strong>Explanation:</strong> As shown you the street at cell (0, 0) is not connected with any street of any other cell and you will get stuck at cell (0, 0)
</pre>

<p><strong class="example">Example 3:</strong></p>

<pre><strong>Input:</strong> grid = [[1,1,2]]
<strong>Output:</strong> false
<strong>Explanation:</strong> You will get stuck at cell (0, 1) and you cannot reach cell (0, 2).
</pre>

<p>&nbsp;</p>
<p><strong>Constraints:</strong></p>

<ul>
	<li><code>m == grid.length</code></li>
	<li><code>n == grid[i].length</code></li>
	<li><code>1 &lt;= m, n &lt;= 300</code></li>
	<li><code>1 &lt;= grid[i][j] &lt;= 6</code></li>
</ul>
</div><span style="font-size: 0px; line-height: 0;">&nbsp;</span></div>

## Initial Intuition

Algorithmically, this problem seems quite simple. Most the complexity appears to lie in implementing the logic. We need to determine where we can go from the current cell from the two possible directions. If the direction is valid, then we move onto that cell. If the direction results in an invalid cell (which would be either outside the array, or a cell that has already been visited), we do not go into that cell. Once we are at the m-1 n-1 index, we can return true. If we run out of valid cells to access, we return false.

This resembles a graph traversal algorithm, so I will use a depth first search and recursion, as I do not feel like using a loop.

One more point to note is that to ensure the current cell is valid, we must check if the direction we entered from is part of one of the directions from the current cell. We should encode the cells and their corresponding directions in a hashmap (or dictionary in python)

### Algorithm

First, define the hashmap as a class variable. In the function, we have to create our visited list, which will store the points that have been visited. I will be using a coordinate system (so (x, y)). Just remember that the outer array for the 2d array is the row number, which is the y, and the inner arrays are the column number, which is the x.

Next is the recursive function. This would take in the current coordinate, grid, and visited array. We check if we are at the goal, which would be when the x == len(grid[0]) - 1 and y = len(grid) - 1. If not, we get which directions we can take from our cell number using the mapping. Using this, we

In [ ]:
from typing import List

class Solution:
    directionMap = {
        1: ['l', 'r'],
        2: ['u', 'd'],
        3: ['l', 'd'],
        4: ['d', 'r'],
        5: ['l', 'u'],
        6: ['u', 'r']
    }

    reverseDirection = {
        'u' : 'd',
        'd' : 'u',
        'r' : 'l',
        'l' : 'r'
    }

    def hasValidPath(self, grid: List[List[int]]) -> bool:
        visited = []
        self.maxX = len(grid[0]) - 1
        self.maxY = len(grid) - 1
        self.maxCoord = (self.maxX, self.maxY)
        return self.recur(grid, (0,0), visited)
        
    def recur(self, grid, coordinate, visited):
        if ((coordinate[0] == self.maxX) and (coordinate[1] == self.maxY)):
            return True

        visited.append(coordinate)
        
        curCell = grid[coordinate[1]][coordinate[0]]
        curDirections = self.directionMap[curCell]

        nextCells = []
        followCells = []
        
        
        for direction in curDirections:
            nextCells.append(self.directionToCoordinate(coordinate, direction))

        for i in range(len(nextCells)):
            curNextCell = nextCells[i]
            curBool = self.coordinateInBounds(curNextCell) # Checks if coordinate is in bounds
            curBool = curBool and not self.coordinateInVisited(curNextCell, visited) # checks if coordinate has been visited
            
            tempBool = False

            if (curBool):
                curNextCellDirections = self.directionMap[grid[curNextCell[1]][curNextCell[0]]] # Gets the directions of the current next cell under examination
                cellDirection = curDirections[i]
                
                if self.reverseDirection[cellDirection] in curNextCellDirections: # Checks if the two cells match up
                    tempBool = True
            
            followCells.append(tempBool)

        result = False

        for i in range(len(nextCells)):
            if followCells[i]:
                result = result or self.recur(grid, nextCells[i], visited)

        return result

    def coordinateInVisited(self, coordinate, visited):
        return coordinate in visited
        
    def coordinateInBounds(self, coordinate):
        if coordinate[0] > self.maxCoord[0] or coordinate[0] < 0:
            return False
        if coordinate[1] > self.maxCoord[1] or coordinate[1] < 0:
            return False
        return True
        
    def directionToCoordinate(self, coordinate, direction):
        if direction == 'l':
            return (coordinate[0] - 1, coordinate[1])
        if direction == 'r':
            return (coordinate[0] + 1, coordinate[1])
        if direction == 'u':
            return (coordinate[0], coordinate[1] - 1)
        if direction == 'd':
            return (coordinate[0], coordinate[1] + 1)
        
        print("ERROR, INVALID SYMBOL")
        return None



In [3]:
s = Solution()

s.hasValidPath([[2,6,3],[6,5,2]])

False

## Result

Unfortunately, this solution has timed out. However, the logic is all fine. Therefore, I decided to use a loop to create a frontier. This would prevent unncessary recursion.

In [31]:
from typing import List

class Solution1:
    directionMap = {
        1: ['l', 'r'],
        2: ['u', 'd'],
        3: ['l', 'd'],
        4: ['d', 'r'],
        5: ['l', 'u'],
        6: ['u', 'r']
    }

    reverseDirection = {
        'u' : 'd',
        'd' : 'u',
        'r' : 'l',
        'l' : 'r'
    }

    def hasValidPath(self, grid: List[List[int]]) -> bool:
        visited = set()
        frontier = [(0,0)]
        lenX = len(grid[0])
        lenY = len(grid)
        
        
        while len(frontier) > 0:
            curCell = frontier.pop()
            visited.add(curCell)
            if curCell[0] == lenX - 1 and curCell[1] == lenY - 1:
                return True
            
            curCellDirs = self.directionMap[self.getFromGridWithCoord(grid, curCell)]

            nextCells = []

            for i in curCellDirs:
                tempCell = [self.directionToCoordinate(curCell, i), i]
                if tempCell[0] not in visited:    
                    nextCells.append(tempCell)

            for j in nextCells:
                if self.coordinateInBounds(grid, j[0]):
                    if self.reverseDirection[j[1]] in self.directionMap[self.getFromGridWithCoord(grid, j[0])]:
                        frontier.append(j[0])
            
        return False

    def getFromGridWithCoord(self, grid, coord):
        return grid[coord[1]][coord[0]]
    
    def coordinateInBounds(self, grid, coord):
        lenX = len(grid[0])
        lenY = len(grid)
        if coord[0] >= 0 and coord[0] < lenX and coord[1] >= 0 and coord[1] < lenY:
            return True
        return False

    def directionToCoordinate(self, coordinate, direction):
        if direction == 'l':
            return (coordinate[0] - 1, coordinate[1])
        if direction == 'r':
            return (coordinate[0] + 1, coordinate[1])
        if direction == 'u':
            return (coordinate[0], coordinate[1] - 1)
        if direction == 'd':
            return (coordinate[0], coordinate[1] + 1)
        
        print("ERROR, INVALID SYMBOL")
        return None
        

In [30]:
s = Solution1()

s.hasValidPath([[4,1],[6,1]])

True

## Result

Unfortunately, this does not work as well. At this point, I was left scratching my head. The size of the input is not too big, and the algorithm should be O(n\*m). Therefore, I resigned, and decided to look at the answer. However, before that, I asked copilot to give me an analyisis of my time commplexity (to ensure I was not going insane). It gave me what I expected: time complexity of O(n\*m). However, it gave me a suggestion to change my visited list into a set. This small change made my solution work! I ended up with a runtime of 286ms, beating 25%. I additionally tried the same with the recursive solution. It also worked, albeit with a runtime of 490ms, beating 11%. 

As always, I am still not satisfied with my solution: how did others get such a better runtime? Even the solution that beat 25% was behind the curve. So I looked at leetcode's editorial solution. One optimisation I could have made is instead of using maths to figure out if two cells can link or not, and having to reverse directions and whatnot, I could have simply had a hashmap indicating which cells link up. This would have saved plenty of time ensuring you can go back from the new cell into the old cell. I additionally learnt about the disjoint set union algorithm, and look forward to be able to use it in the future!

I additionally looked at an interesting solution by Eunice (https://leetcode.com/problems/check-if-there-is-a-valid-path-in-a-grid/solutions/8100735/deterministic-path-validation-state-transition-matrix-beats-100/?envType=daily-question&envId=2026-04-27)

In this solution, Eunice used a state transition table to determine a path. Additionally, they used the following optimisations: 

 - if the starting cell is a 5 (j shaped piece), there is no solution
 - if the ending cell is a 4 (r shaped piece), there is no solution
 - we do not need to keep track of the visited nodes as the only time we would retrace is if there was a loop, and that only occurs if the starting cell is a 4 (r shaped piece), so we would only ever need to track if we retrace the starting cell. If we do (before reaching the ending cell), there is no solution

 Not only was Eunice's solution much cleaner, but also so much more efficient, despite the similar time complexity. They mention however, that this state transition table technique can only be used when we have a definite start and end, and the graph must have static connectivity.